# CS 555 Spring 2024 - Final project
# Solving Advection Equations on Spherical Geodesic Grids
### Wenhan Tang

## Introduction
This project addresses the advection equation using a spherical geodesic mesh grid. The advection algorithm in the mini-app is used for solveing the 2-dimensional advection euqation:
\begin{equation}
    \frac{\partial \rho}{\partial t} + \nabla \cdot (\vec{V} \rho) = \dot{S},
\end{equation}
where $\rho$ represents any extensive quantity in unit of volume (e.g., density of air, concentration of pollution, thermal energy per unit of volume ...), $\vec{V}$ is the 2D wind, and $\dot{S}$ indicates the source of the $\rho$.

### The goals of this mini-app

1. Develop a versatile tool for generating spherical triangle or hexagon mesh grids at various resolutions.
Create 3D visualization tools for displaying variables on the spherical surface.

2. Implement a first-order upstream scheme based on the finite-volume method to solve the advection equation, ensuring mass (or energy) conservation.

3. Simulate global temperature evolution at the 500 hPa level (approximately 5800 m altitude) using real wind data, and compare the results with actual temperature measurements.

4. Model the distribution of CO2 concentrations emitted during an airplane's flight.

### Algorithm used for advection simulation

A first-order upwind scheme based on finite-volume method is used in this mini-app, the mathematical expression is

\begin{equation}
    \rho_{i}^{t+1} = \rho_{i}^{t} - \frac{1}{S_{i}H}\sum_{j=1}^{N_i} f_{i,j}^{t} \Delta t,
\end{equation}
\begin{equation}
    f_{i,j}^{t} = V_{i,j} L_{i,j} H \cdot \left(\delta \left(V_{i,j} \geq 0 \right) \rho_i^t + \delta \left(V_{i,j} < 0 \right)  \rho_j^t \right),
\end{equation}

where $\rho_{i}^{t}$ and $\rho_{i}^{t+1}$ denote the values of $\rho$ in the $i^{th}$ tail at times $t$ and $t+1$, respectively. $N_i$ is the number of neighbor tails of the $i^{th}$ tail (for triangle grid, it is 3, for pentagon grid, it is 5, for hexagon grid, it is 6). The term $f_{i,j}^{t}$ represents the flux from tail $i$ to tail $j$. $V_{i,j}$ is the wind speed projected onto the normal direction of the boundary line between tail $i$ and tail $j$, with the direction from $i$ to $j$ considered positive. $S_{i}$ is the area of the $i^{th}$ tail, while $H$ is a constant value that defines the vertical height of the 2D domain in the simulation. $\Delta t$ is the time step, $L_{i,j}$ is the length of the boundary between tail $i$ and $j$. $\delta(\cdot)$ is an operator that returns a value of 1 if the enclosed expression is true, and 0 otherwise.

### Overview of this Jupyter notebook

$\cdot$ Section 0 covers the installation process.

$\cdot$ Section 1 focuses on generating the spherical mesh grid, saving it to a file, and its visualization.

$\cdot$ Section 2 presents ideal tests to evaluate the advection scheme by comparing numerical solutions with exact solutions.

$\cdot$ Section 3 displays the simulation results for global temperature advection, including verification of mass conservation.

$\cdot$ Section 4 demonstrates the simulation results of the CO2 plume emitted by an airplane.

$\cdot$ Section 5 is the reflection, including my learning, limitations, and my potential future works.

The core code is stored in the src/ folder as a Python package. All data required by the mini-app are located in the data/ folder. The meshgrid/ folder, initially empty, is designated for storing mesh grid files generated in Section 1. The plots/ folder holds the temporary figures created in Sections 2-4.

__FYI: I have tested this mini-app on two computers and the Linux system, and it runs well without any bugs. If you encounter any bugs on your computer and system, please contact me via email (wenhant2@illinois.edu) or Slack, and I will be happy to help you resolve the issues.__



## 0.Installation

Make sure that this jupyter notebook is located in a folder which is prepared specially for Wenhan's final project. The python packages below are needed for running the mini-app:

In [ ]:
# Check whether the python packages below are successfully installed.
import numpy
import scipy
import netCDF4
import matplotlib
import plotly
import PIL
import cartopy
#import shapefile
#import shapely
import IPython
import json
import os

The kaleido package is necessary for converting 3D plots (by plotly) into static images. It can be easily installed with the command "pip install -U kaleido". If kaleido is not already installed, please use this command to install it.

In [ ]:
try:
    import kaleido
except ModuleNotFoundError:
    print("No kaleido packages, install it.")
    !pip install -U kaleido
    import kaleido

Check whether the core code and data files are located in specific directories. If any error occurs, please contact Wenhan Tang by wenhant2@illinois.edu.

In [ ]:
assert os.path.exists("src"), "Please contact Wenhan Tang (wenhant2@illinois.edu) about this issue."
assert os.path.exists("src/advection.py"), "Please contact Wenhan Tang (wenhant2@illinois.edu) about this issue."
assert os.path.exists("src/colorscale.py"), "Please contact Wenhan Tang (wenhant2@illinois.edu) about this issue."
assert os.path.exists("src/data_loading.py"), "Please contact Wenhan Tang (wenhant2@illinois.edu) about this issue."
assert os.path.exists("src/experiment.py"), "Please contact Wenhan Tang (wenhant2@illinois.edu) about this issue."
assert os.path.exists("src/ideal_scenarios.py"), "Please contact Wenhan Tang (wenhant2@illinois.edu) about this issue."
assert os.path.exists("src/real_scenarios.py"), "Please contact Wenhan Tang (wenhant2@illinois.edu) about this issue."
assert os.path.exists("src/show_progress.py"), "Please contact Wenhan Tang (wenhant2@illinois.edu) about this issue."
assert os.path.exists("src/sphere_mesh.py"), "Please contact Wenhan Tang (wenhant2@illinois.edu) about this issue."
assert os.path.exists("dirs.py"), "Please contact Wenhan Tang (wenhant2@illinois.edu) about this issue."

assert os.path.exists("data"), "Please contact Wenhan Tang (wenhant2@illinois.edu) about this issue."
assert os.path.exists("data/ERA5-pl-global.nc"), "Please contact Wenhan Tang (wenhant2@illinois.edu) about this issue."
assert os.path.exists("data/coastlines.json"), "Please contact Wenhan Tang (wenhant2@illinois.edu) about this issue."
assert os.path.exists("data/landocean_hexagon_order4.npy"), "Please contact Wenhan Tang (wenhant2@illinois.edu) about this issue."
assert os.path.exists("data/landocean_hexagon_order5.npy"), "Please contact Wenhan Tang (wenhant2@illinois.edu) about this issue."
assert os.path.exists("data/landocean_hexagon_order6.npy"), "Please contact Wenhan Tang (wenhant2@illinois.edu) about this issue."

from dirs import *
if not(os.path.exists(meshDir)):
    os.makedirs(meshDir)
    print("Create directory: " + meshDir)
    
if not(os.path.exists(plotDir)):
    os.makedirs(plotDir)
    print("Create directory: " + plotDir)

If the first two sections of code execute successfully, then the installation process of this mini-app is complete.

## 1. Make Spherical Mesh Grids

$\cdot$ This section is used for creating the spherical hexagon meshgrid and save to file.

$\cdot$ The code also save the parent spherical trangle meshgrids and several hexagon meshgrids in different order.

$\cdot$ The generation of a spherical triangle meshgrid is started from a icosahedron, then the bisection method is used to increase the resolution of the mesh. The spherical hexagon meshgrid is converted from spherical triangle meshgrid.

$\cdot$ For more details about spherical triangle/hexagon meshgrid, please see src/sphere_mesh.py

In [ ]:
from src.sphere_mesh import SphereTriMesh, SphereHexMesh, create_grid
from dirs import *

import numpy as np

### 1.1 Set parameters

$\cdot$ R: Radius of Earth (units: m).

$\cdot$ order: The times of applying the bisection algorithm, larger the value, higher the resolution of the meshgrid.

$\cdot$ grid_type: "triangle" or "hexagon" (The advection algorithm in this project is specifically designed for hexagon).

$\cdot$ do_create_grid: For the first time run, set do_create_grid = True, when meshgrid files already exist, set to False.

In [ ]:
R = 6371000 # Radius of Earth
order = 6 # Order of the mesh grids
grid_type = "hexagon" # Type of the geodesic mesh grid, "triangle" or "hexagon"

### 1.2 Generate the mesh grids

It may take a little while (~13 minutes on my computer), but only need to run it once, as the results will be saved to netCDF files in meshgrid/.

If the meshgrid files in the ./meshgrid directory already exist, set do_create_grid = False to skip this step.

Once you see "!! Successfully Complete !!", it is done.

In [ ]:
if not(os.path.exists(meshDir + "/hexagon_order" + str(order) + ".nc")):
    do_create_grid = True # For the first time
else:
    do_create_grid = False # For furthur running

if do_create_grid:
    objMesh = create_grid(
        R = R,
        order = order,
        save_history_to_file = True,
        save_history_hexagon_to_file = True,
        fileDir = meshDir,
        grid_type = grid_type,
    )
    objMesh.save_to_file(meshDir + "/" + grid_type + "_order" + str(order) + ".nc")
    assert os.path.exists(meshDir + "/" + grid_type + "_order" + str(order) + ".nc")
    print("!! Successfully Complete !!")
    del(objMesh)
else:
    print("The mesh file already exists, skip ...")

In [ ]:
"""
    Used for open the meshgrid file and loading the meshgrid.
"""
def load_mesh(order, grid_type):
    if grid_type == "triangle":
        objMesh = SphereTriMesh(from_file = True, fileName = meshDir + "/triangle_order" + str(order) + ".nc")
    elif grid_type == "hexagon":
        objMesh = SphereHexMesh(from_file = True, fileName = meshDir + "/hexagon_order" + str(order) + ".nc")
    else:
        assert False, "Should be one of \"triangle\" or \"hexagon\"."
    return objMesh

### 1.3 Visualization samples for Spherical mesh

__(All the visual outputs in this section are 3D views, which can be freely zoomed and rotated to view from different angles)__

#### 1.3.1 Spherical triangle mesh grid

To save the run time, only the 2$^{nd}$ order triangle mesh grid and the hexagon mesh which is converted from the 2$^{nd}$ order triangle mesh grid are shown here. The yellow points are the vertices of this mesh grid, and the red lines are the edges of each triangle tail.

In [ ]:
objMesh = load_mesh(order = 2, grid_type = "triangle")
fig = objMesh.plot(title = "2 order triangle mesh grid")
fig.show()

#### 1.3.2 Spherical hexagon mesh grid
This is a 2$^{nd}$ order hexagon mesh grid converted by the 2$^{nd}$ order triangle mesh grid. The yellow points are the vertices of this mesh grid, and the red lines are the edges of each triangle tail. Noting that there are 12 pantagon tails on the spherical surface, while all other tails are hexagons.

In [ ]:
objMesh = load_mesh(order = 2, grid_type = "hexagon")
fig = objMesh.plot(title = "2 order hexagon mesh grid")
fig.show()

### 1.4 Staggered grid for physical variables and flux
The 3D plot below shows the staggered grid on 2$^{nd}$ order spherical hexagon meshgrid. The green dots are located in the center of each hexagon (pentagon) grid cell, which stores the data of physical variables (temperature, pollution concentration, ...). The blue dots are located in the center of each edge, which stores the data of wind (u, v components) for computing the fluxes among grid cells.

In [ ]:
fig = objMesh.plot_grid(title = "Staggered grid on 2 order spherical hexagon meshgrid")
fig.show()

### 1.5 Visualization of 6 order hexagon mesh grid

In this final project, all advection simulation are conducted on a 6$^{th}$ order hexagon mesh grid. However, it is inefficient to show its cells, edges, and grid points on that mesh grid. The next jupyter notebook cell displays a visualization demo of the 6$^{th}$ order hexagon mesh grid, showing the spatial resolution of this mesh grid. Grey represents land region, while blue represents ocean region.

In [ ]:
objMesh = load_mesh(order = 6, grid_type = "hexagon")
objMesh.implement_landocean_fromnpy(dataDir + "/landocean_hexagon_order6.npy")
fig = objMesh.plot_landocean(title = "Land and ocean displayed on 6 order spherical hexagon meshgrid")
fig.show()

### 1.6 Conclusion

In this section, spherical triangle and hexagon mesh grids ranging from order 1 to 6 have been created. The spherical 3D visualization tools have been employed to display the vertices, edges, tiles, and data locations on the 2nd triangle and hexagon mesh grids. The final figure illustrates the resolution of the 6th hexagon mesh grid, demonstrating how the visualization tools represent data distributed on a spherical surface.

The results presented in this section confirm that the first objective outlined in the introduction has been achieved:  __"Develop a versatile tool for generating spherical triangle or hexagon mesh grids at various resolutions. Create 3D visualization tools for displaying variables on the spherical surface."__

## 2. Ideal Scenarios Tests

This section is used for testing the 1 order advection scheme. Two ideal pollution advection simulations are shown in this section. 

In [ ]:
# Import packages
from src.experiment import Experiment
from dirs import *
import numpy as np

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import Image, display

### 2.1 Ideal Scenario 1: Ractangle pollution in easterly wind

In this scenario, the initail distribution of pollution is within a "ractangle" in lat-lon coordinates, flow with an easterly wind.

$\cdot$ Initial distribution: 30˚E - 90˚E, 15˚N - 55˚N

$\cdot$ Initial concentration: 1 Kg/m$^3$

$\cdot$ Wind u component: $-463.31 \cdot cos(\varphi)$ (units: m/s)

$\cdot$ Wind v component: 0 m/s

(So that every point has the same angular velocity: $2\pi /day$)

$\cdot$ Source: 0 Kg/m$^2$/s (i.e., no source and sink)

$\cdot$ Simulation time: 24 hours

After 24 hours, the pollution "ractangle" should return back to the initial place with the same shape, therefore, the initail condition of pollution concentration can also be regarded as the "exact" solution after 24 hours. 

#### 2.1.1 Run the simulation

In [ ]:
"""
    Create the simulation experiment.
"""
meshFile = meshDir + "/hexagon_order6.nc"

objExp = Experiment(
    scenario_name = "Ractangle_Block",
    scenario_type = "ideal",
    meshFile = meshFile,
    solver_name = "AdvectionSolver",
    kwargs_Scenario = {
        "lon1": 30,
        "lon2": 90,
        "lat1": 15,
        "lat2": 55,
        "wind": 463.31, # Maximum wind speed (units: m/s)
        "pollution_conc": 1.0, # Uniform pollution concentration (units: Kg/m^3)
    },
)

In [ ]:
"""
    Run the simulation.
"""
results = objExp.simulation(
    total_time = 24, # Total simulation time = 24 hours 
    timeUnit = "hr", # Time unit is hour
    courant = 0.8, # Courant number is set to 0.8
)

In [ ]:
"""
    The function searches for the nearest simulation results corresponding to a specific time.
"""
def search_nearest_itime(results, time):
    return np.argmin(np.abs(results["timeList"] - time))

#### 2.1.2 Draw the initial (exact solution) and last step (numerical solution) of pollution concentration results

The results shown below indicates the advection algorithm is able to simulate the main phenomenon of the advection. The high-pollutant regions between exact and numerical solution are nearly the same. There is a strong "diffuse" phenomenon in the numerical solution, and the "dissipation" phenomenon is significant.

In [ ]:
"""
    Visualization function.
"""
def draw_distribution(data, objExp, outName, cmin = 0, cmax = 1):
    if not(objExp.objMesh.coastlines_implemented):
        objExp.objMesh.implement_coastlines(dataDir + "/coastlines.json")
    fig = objExp.objMesh.plot_data(
        #fig = fig,
        data = data,
        colorscale = "Greens",
        opacity = 1,
        showscale = True,
        cmin = cmin, cmax = cmax,
        #row = 1, col = 1,    
    )
    fig = objExp.objMesh.plot_coastlines(fig, threshold = 10, color = "black")
    fig = objExp.objMesh.plot_gridlines(fig)
    fig.update_layout(margin=dict(l=0, r=0, t=0, b=0))
    fig.write_image(outName, width=960, height=540, scale=3)
    print("Image is saved to " + outName)
    #return fig

In [ ]:
"""
    Draw the numerical and exact solution and save as png files.
"""
exact_data = results["var_show_dict"]["pollution"][0]
numerical_data = results["var_show_dict"]["pollution"][search_nearest_itime(results, 24 * 3600)]
draw_distribution(exact_data, objExp, outName = plotDir + "/Ractangle_Block_exact.png")
draw_distribution(numerical_data, objExp, outName = plotDir + "/Ractangle_Block_numerical.png")

In [ ]:
"""
    Display the results.
"""
fig = plt.figure(figsize = (25, 12))
ax1 = fig.add_subplot(1, 2, 1)
img = mpimg.imread(plotDir + "/Ractangle_Block_exact.png")
ax1.imshow(img)
ax1.axis("off")
ax1.set_title("Exact solution", fontsize = 20)
ax2 = fig.add_subplot(1, 2, 2)
img = mpimg.imread(plotDir + "/Ractangle_Block_numerical.png")
ax2.imshow(img)
ax2.axis("off")
ax2.set_title("Numerical solution", fontsize = 20)
plt.tight_layout(pad=0.0)
plt.show()

#### 2.1.3 Cheking the mass conservation

In [ ]:
total_mass_init = results["var_ext_dict"]["pollution"][0].sum()
total_mass_final = results["var_ext_dict"]["pollution"][-1].sum()
print("Total mass at initial condition is %.2f ton." % (total_mass_init * 1e-4))
print("Total mass at the last time step is %.2f ton." % (total_mass_final * 1e-4))
print()
if np.abs(total_mass_init - total_mass_final) < 1e-2:
    print("Satisfies mass conservation!")
else:
    print("Does not satisfy mass conservation!")

### 2.2 Ideal Scenario 2: Ractangle pollution in meridional wind

In this scenario, the initail distribution of pollution is within a "ractangle" in lat-lon coordinates, flow with an meridional wind that blows from the south to the north in the Eastern Hemisphere and from north to south in the Western Hemisphere.

$\cdot$ Initial distribution: 30˚E - 90˚E, 15˚N - 55˚N

$\cdot$ Initial concentration: 1 Kg/m$^3$

$\cdot$ Wind u component: $-463.31 \cdot cos(\lambda) cos(\varphi)$ (units: m/s)

$\cdot$ Wind v component: $463.31 \cdot sin(\lambda)$ m/s

(So that every point has the same angular velocity: $2\pi /day$)

$\cdot$ Source: 0 Kg/m$^2$/s (i.e., no source and sink)

$\cdot$ Simulation time: 24 hours

In this scenario, the pollution "ractangle" will move across both the north polar and the south polar. After 24 hours, the pollution "ractangle" should return back to the initial place with the same shape, therefore, the initail condition of pollution concentration can also be regarded as the "exact" solution after 24 hours.

In [ ]:
"""
    Create the simulation experiment.
"""
objExp = Experiment(
    scenario_name = "Ractangle_Block_CrossPolar",
    scenario_type = "ideal",
    meshFile = meshFile,
    solver_name = "AdvectionSolver",
    kwargs_Scenario = {
        "lon1": 30,
        "lon2": 90,
        "lat1": 15,
        "lat2": 55,
        "wind": 463.31, # Maximum wind speed (units: m/s)
        "pollution_conc": 1.0, # Uniform pollution concentration (units: Kg/m^3)
    },
)

#### 2.2.1 Run the simulation

In [ ]:
"""
    Run the simulation.
"""
results = objExp.simulation(
    total_time = 24, # Total simulation time = 24 hours 
    timeUnit = "hr", # Time unit is hour
    courant = 0.8, # Courant number is set to 0.8
)

#### 2.2.2 Draw the initial (exact solution) and last step (numerical solution) of pollution concentration results

The results shown below indicates the advection algorithm is able to simulate the main phenomenon of the advection. The high-pollutant regions between exact and numerical solution are nearly the same. There is a strong "diffuse" phenomenon in the numerical solution, and the "dissipation" phenomenon is significant.

In [ ]:
"""
    Draw the numerical and exact solution and saving as png files.
"""
exact_data = results["var_show_dict"]["pollution"][0]
numerical_data = results["var_show_dict"]["pollution"][search_nearest_itime(results, 24 * 3600)]
draw_distribution(exact_data, objExp, outName = plotDir + "/Ractangle_Block_CrossPolar_exact.png")
draw_distribution(numerical_data, objExp, outName = plotDir + "/Ractangle_Block_CrossPolar_numerical.png")

In [ ]:
"""
    Display the results.
"""
fig = plt.figure(figsize = (25, 12))
ax1 = fig.add_subplot(1, 2, 1)
img = mpimg.imread(plotDir + "/Ractangle_Block_CrossPolar_exact.png")
ax1.imshow(img)
ax1.axis("off")
ax1.set_title("Exact solution", fontsize = 20)
ax2 = fig.add_subplot(1, 2, 2)
img = mpimg.imread(plotDir + "/Ractangle_Block_CrossPolar_numerical.png")
ax2.imshow(img)
ax2.axis("off")
ax2.set_title("Numerical solution", fontsize = 20)
plt.tight_layout(pad=0.0)
plt.show()

#### 2.2.3 Cheking the mass conservation

In [ ]:
total_mass_init = results["var_ext_dict"]["pollution"][0].sum()
total_mass_final = results["var_ext_dict"]["pollution"][-1].sum()
print("Total mass at initial condition is %.2f ton." % (total_mass_init * 1e-4))
print("Total mass at the last time step is %.2f ton." % (total_mass_final * 1e-4))
print()

if np.abs(total_mass_init - total_mass_final) < 1e-2:
    print("Satisfies mass conservation!")
else:
    print("Does not satisfy mass conservation!")

### 2.3 Conclusion and Discussion

In this section, two ideal experiments separately simulated the zonal and meridional circulation of pollutants around the Earth. In the simulation results, the flow speed of the pollutants is reasonable, and their final positions meet expectations. Generally, handling simulations near the poles is a challenge in global atmospheric modeling, and the second experiment successfully modeled the process of pollutants passing over the North and South Poles. The simulation results also follow the mass conservation. However, the simulation results showed a noticeable dissipation phenomenon (i.e., false diffusion), and using higher-order numerical schemes or more precise grids could help reduce this effect.

The results presented in this section confirm that the second objective outlined in the introduction has been achieved:  __"Implement a first-order upstream scheme based on the finite-volume method to solve the advection equation, ensuring mass (or energy) conservation."__

## 3. Global Temperature Advection

In this section, we use a first-order upwind scheme to simulate the advection of global atmospheric temperature at the 500 hPa level. The simulation covers a 24-hour period from 0:00 UTC on January 1, 2013, to 0:00 UTC on January 2, 2013. We assume that the atmosphere is an ideal gas (i.e., $pV=nRT$), and initial conditions have the 500 hPa isobaric surface at the same height. During the advection process, we ignore vertical atmospheric movements and exclude both adiabatic and non-adiabatic heating/cooling. The atmosphere should satisfies both mass and thermal energy conservation.

The equations we solved for the temperature advection are:
\begin{equation}
    C_v \frac{\partial \rho T}{\partial t} + C_v \nabla \cdot (\vec{V} \rho T) = 0;
\end{equation}
\begin{equation}
    \frac{\partial \rho}{\partial t} + \nabla \cdot (\vec{V} \rho) = 0.
\end{equation}

The first equation is the thermal energy conservative equation, while the second is the mass conservation equation. $C_v$ represents the constant-volume specific heat of air, while $\rho$ and $T$ denote the density and temperature of the air, respectively.

We use temperature data at 0:00 UTC on January 1, 2013 for the initial condition, and the hourly wind data from January 1 to January 2 for the boundary condition. All the data are retrieved from European Centre for Medium-Range Weather Forecasts (ECMWF) Reanalysis Dataset (website: https://cds.climate.copernicus.eu/cdsapp#!/search?type=dataset).

In [ ]:
# Import packages
from src.experiment import Experiment
from src.real_scenarios import Scenario_Temperature_Advection
from src.sphere_mesh import SphereHexMesh
from src.colorscale import get_colormap
from dirs import *

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image as PIL_Image
from IPython.display import Image, HTML

### 3.1 Run the simulation

In [ ]:
"""
    Create the simulation experiment.
"""

meshName = "hexagon_order6"
meshFile = meshDir + "/" + meshName + ".nc"

scenario_name = "Temperature_Advection"
#scenario_type = "ideal"
scenario_type = "real"
solver_name = "AdvectionSolver"

objExp = Experiment(
    scenario_name = "Temperature_Advection",
    scenario_type = "real",
    meshFile = meshFile,
    solver_name = "AdvectionSolver",
    kwargs_Scenario = {
        "dataDir": dataDir,
    }

)
objExp.objMesh.implement_landocean_fromnpy(dataDir + "/landocean_" + meshName + ".npy")

In [ ]:
"""
    Run the simulation.
    (It takes ~3 minutes on my computer)
"""
results = objExp.simulation(total_time = 24, timeUnit = "hr", courant = 0.8)

### 3.2 Mass and energy conservation

The results below show the left graph depicting the change in total atmospheric thermal energy over simulation time, and the right graph depicting the change in total mass over simulation time. Clearly, both the total thermal energy and total mass of the atmosphere remain constant over time, indicating that the simulation results satisfy both mass conservation and thermal energy conservation.

In [ ]:
Cv = 717 # constant-volume specific heat of air (unit: J/Kg/k)
densT_V_timeList = results["var_ext_dict"]["densT"].sum(axis = 1) # rho * V * T = m * T
energy_timeList = Cv * densT_V_timeList # m * Cv * T, i.e., the thermal energy
mass_timeList = results["var_ext_dict"]["dens"].sum(axis = 1) # rho * V, i.e., the mass
timeList = results["timeList"]
fig = plt.figure(figsize = (9, 4))
ax1 = fig.add_subplot(1, 2, 1)
ax1.plot(timeList / 3600, energy_timeList, color = "red")
ax1.grid(linestyle = "dashed")
ax1.set_xlabel("Time (hour)")
ax1.set_ylabel("Energy (J)")
ax1.set_title("Total thermal energy")

ax2 = fig.add_subplot(1, 2, 2)
ax2.plot(timeList / 3600, mass_timeList, color = "blue")
ax2.grid(linestyle = "dashed")
ax2.set_xlabel("Time (hour)")
ax2.set_ylabel("Mass (Kg)")
ax2.set_title("Total mass")

plt.show()

### 3.3 Results visualization

In [ ]:
"""
    The function searches for the nearest simulation results corresponding to a specific time.
"""
def search_nearest_itime(results, time):
    return np.argmin(np.abs(results["timeList"] - time))

In [ ]:
"""
    Draw the numerical solution at a specific time and save as a png file.
"""
def draw_one_time_for_ta(time, objExp, results, outName):
    
    cmap = get_colormap("jet")

    densT_alltime = results["var_show_dict"]["densT"]
    dens_alltime = results["var_show_dict"]["dens"]
    T_alltime = densT_alltime / dens_alltime
    Temperature = T_alltime[search_nearest_itime(results, time)]
    
    T_max = T_alltime.max()
    T_min = T_alltime.min()

    fig = objExp.objMesh.plot_data(
        data = Temperature,
        #colorscale = "jet",
        #colorscale = "Greens",
        colorscale = cmap,
        opacity = 1,
        showscale = True,
        cmin = T_min,
        cmax = T_max,
    )
    if not(objExp.objMesh.coastlines_implemented):
        objExp.objMesh.implement_coastlines(dataDir + "/coastlines.json")
    fig = objExp.objMesh.plot_coastlines(fig, threshold = 10, color = "black")
    fig = objExp.objMesh.plot_gridlines(fig)
    #fig.write_image(plotDir + "/airplane_emission.png", width=960, height=540, scale=2)
    fig.update_layout(title = "Temperature (k) at 500 hPa; time = " + str(time / 3600) + " hr",margin=dict(l=0, r=0, t=25, b=0))  # 设置左、右、顶、底的边距
    fig.write_image(outName, width=960, height=540, scale=3)
    print("Image is saved to " + outName)

In [ ]:
"""
    Draw a real gloabl temperature distribution (from ECMWF data) at a specific time and save as a png file.
"""
def draw_one_time_for_realT(time, objExp, outName, cmin = None, cmax = None):
    
    cmap = get_colormap("jet")

 
    #i_time = search_nearest_itime(objExp.objScn, time)
    realT = objExp.objScn.get_netcdf_data_anyTime(
        time = time,
        varFileName = "t",
        objMesh = objExp.objMesh, 
        lon = objExp.objMesh.lon_c,
        lat = objExp.objMesh.lat_c
    )
    
    realT_max = realT.max()
    realT_min = realT.min()
    
    if cmin is None:
        T_min = realT_min
    else:
        T_min = np.min([realT_min, cmin])
    if cmax is None:
        T_max = realT_max
    else:
        T_max = np.max([realT_max, cmax])

    fig = objExp.objMesh.plot_data(
        data = realT,
        #colorscale = "jet",
        #colorscale = "Greens",
        colorscale = cmap,
        opacity = 1,
        showscale = True,
        cmin = T_min,
        cmax = T_max,
    )
    if not(objExp.objMesh.coastlines_implemented):
        objExp.objMesh.implement_coastlines(dataDir + "/coastlines.json")
    fig = objExp.objMesh.plot_coastlines(fig, threshold = 10, color = "black")
    fig = objExp.objMesh.plot_gridlines(fig)
    #fig.write_image(plotDir + "/airplane_emission.png", width=960, height=540, scale=2)
    fig.update_layout(title = "Temperature (k) at 500 hPa; time = " + str(time / 3600) + " hr",margin=dict(l=0, r=0, t=25, b=0))  # 设置左、右、顶、底的边距  
    fig.write_image(outName, width=960, height=540, scale=3)
    print("Image is saved to " + outName)

In [ ]:
"""
    Plot the global temperature simulation result at each hour.
    (It may take a little while, ~8 minutes on my computer)
"""
hourPlots = np.arange(25) #[0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24]
for hour in hourPlots:
    print("Drawing hour = ", hour, " ...")
    time = hour * 3600
    outName = plotDir + "/temperature_advection_t_" + str(hour).zfill(3) + ".png"
    draw_one_time_for_ta(time, objExp, results, outName)
print("!! Complete !!")

In [ ]:
"""
   Create a gif file showing the animation of global temperature simulation results.
"""
hourPlots = np.arange(25) #[0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24]
images = [PIL_Image.open(plotDir + "/temperature_advection_t_" + str(hour).zfill(3) + ".png") for hour in hourPlots]
images[0].save(plotDir + '/temperature_advection.gif', save_all=True, append_images=images[1:], optimize=False, duration=500, loop=0)
Image(url=plotDir + '/temperature_advection.gif')

#HTML("<img src=\"" + plotDir + "/temperature_advection.gif\" style=\"width:960px; height:540px;\"/>")

### 3.4 Comparison between simulation results and real data

Here, the temperature simulation results are compared with the real simulation data retrieved from ECMWF reanalysis dataset at 00:00, 06:00, 12:00, 18:00, 24:00 (UTC) on January 1, 2013. The left panel shows the simulation results while the right panel shows the real temperature data.

In [ ]:
densT_alltime = results["var_show_dict"]["densT"]
dens_alltime = results["var_show_dict"]["dens"]
T_alltime = densT_alltime / dens_alltime
cmax = T_alltime.max()
cmin = T_alltime.min()

hourRealPlots = [0, 6, 12, 18, 24]
for hour in hourRealPlots:
    print("Drawing real temperature; hour = ", hour, " ...")
    time = hour * 3600
    outName = plotDir + "/temperature_advection_realT_t_" + str(hour).zfill(3) + ".png"
    draw_one_time_for_realT(time, objExp, outName, cmin = cmin, cmax = cmax)

In [ ]:
image_paths = []
hourList = []
hourRealPlots = [0, 6, 12, 18, 24]
for hour in hourRealPlots:
    hourList += [hour, hour]
    image_paths.append(plotDir + "/temperature_advection_t_" + str(hour).zfill(3) + ".png")
    image_paths.append(plotDir + "/temperature_advection_realT_t_" + str(hour).zfill(3) + ".png")

nLine = len(hourRealPlots)
fig, axs = plt.subplots(nLine, 2, figsize=(20, nLine * 10))

for i, (hour, ax) in enumerate(zip(hourList, axs.flat)):
    img = mpimg.imread(image_paths[i])
    ax.imshow(img)
    ax.axis('off')
    if np.mod(i, 2) == 0:
        ax.set_title("Simulation temperature (k); time = " + str(hour) + " hr", fontsize = 20)
    else:
        ax.set_title("Real temperature (k); time = " + str(hour) + " hr", fontsize = 20)

plt.tight_layout(pad=0.0)
plt.show()

### 3.5 Conclusion and Discussion

This section presents a simulation of the global advection of 500 hPa temperature. The overall simulation results are reasonable and capture the fundamental characteristics of atmospheric advection. The movement direction of cold and warm air masses is generally consistent with real atmospheric temperature data.

It is important to note that the real temperature data is not the exact solution to the advection equation in this problem, because the advection equation is based on highly simplified physical processes. In addition to advection, real atmospheric temperatures are influenced by factors such as vertical atmospheric movements, adiabatic and non-adiabatic heating or cooling, and terrain effects. The discrepancies between the simulated and real temperatures are mainly due to these unconsidered factors. For example, the weakening of the cold vortex in the northeastern Caspian Sea in the simulation results, versus its intensification in real data, is caused by a series of complex thermodynamic and dynamic physical processes; considering only advection is insufficient. Additionally, in the simulation results, a cold trough advected to the northern part of the Tibetan Plateau, but due to the plateau's heating effect on the mid-level atmosphere, the cold trough turns into a warm ridge in real temperature data. However, the dissipation phenomenon in the advection scheme also contributes to the weakening of the cold vortex in the northeastern Caspian Sea, and using higher-order schemes or increasing grid resolution could help improve simulation accuracy.

The results presented in this section confirm that the third objective outlined in the introduction has been achieved: __"Simulate global temperature evolution at the 500 hPa level (approximately 5800 m altitude) using real wind data, and compare the results with actual temperature measurements."__

## 4. Airplane Emission

In this experiment, an airplane departures from one city at 00:00 (UTC) on January 1$^{st}$ 2013, and arrive to another city at 00:00 (UTC) on January 2$^{nd}$ 2013. Now let's simulate the advection of CO$_2$ it emits. The plane is assumed to fly at the 500hPa level along a great circle path, the takeoff and landing processes are ignored. The speed is a constant that let the plane arrive to Munich within just 24 hours.

The advection equation solved in this problem is
\begin{equation}
    \frac{\partial \rho_{CO_2}}{\partial t} + \nabla \cdot (\vec{V} \rho_{CO_2}) = \dot{S}_{emit},
\end{equation}
where $\rho_{CO_2}$ denotes the density of CO$_2$ and $\dot{S}_{emit}$ represents the CO$_2$ emission rate of the airplane. $\vec{V}$ is the 2D wind vector, which are derived from ECMWF reanalysis dataset in this program.

In [ ]:
# Import packages
from src.experiment import Experiment
from src.real_scenarios import Scenario_Airplane_Emission
from src.sphere_mesh import SphereHexMesh
from src.colorscale import get_colormap
from dirs import *

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from PIL import Image as PIL_Image
from IPython.display import Image, HTML

### 4.1 Set parameters

#### 4.1.1 Set the departure/arrival cites and the CO2 emission rate of airplane

In [ ]:
"""
    Departure city
"""
#lon1, lat1 = 75.9, 39.42 # Kashgar,China
#lon1, lat1 = 116, 41 # Beijing, China
lon1, lat1 = 106.65, 29.71 # Chongqin, China
#lon1, lat1 = 121.33, 31.20 # Shanghai, China

"""
   Arrival city
"""
lon2, lat2 = 11.58, 48.13 # Munich,German

"""
   CO2 emission rate
"""
airplane_co2_emission = 2.19 # Kg / s

In [ ]:
meshName = "hexagon_order6"
meshFile = meshDir + "/" + meshName + ".nc"

objExp = Experiment(
    scenario_name = "Airplane_Emission",
    scenario_type = "real",
    meshFile = meshFile,
    solver_name = "AdvectionSolver",
    kwargs_Scenario = {
        "dataDir": dataDir,
        "lon1": lon1,
        "lat1": lat1,
        "lon2": lon2,
        "lat2": lat2,
        "time1": 0,
        "time2": 24 * 3600,
        "co2_emission": airplane_co2_emission,
    }
)
objExp.objMesh.implement_landocean_fromnpy(dataDir + "/landocean_" + meshName + ".npy")

#### 4.1.2 Visulization of airplane path

The red line shows the airplane path, which is a great circle path between Chongqin in China and Munich in German.

__This section of code may involve downloading some small geographic data from the internet for plotting. If there are issues with the internet connection, it might affect the execution of this code or the final plotting result. If this is the case, you can just skip this section and proceed directly to the next one.__

In [ ]:
pathLon, pathLat = objExp.objScn.get_great_circle_path()
fig = plt.figure(figsize = (10, 5))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax.set_extent([-180, 180, -90, 90])
ax.add_feature(cfeature.LAND)
ax.add_feature(cfeature.OCEAN)
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS, linestyle=':')
ax.plot(pathLon, pathLat, 'r-', transform=ccrs.Geodetic(), label='Airplane Path')
ax.legend(loc='best')
#ax.plot()
plt.show()

### 4.2 Run the simulation

In [ ]:
"""
    Run the simulation.
    (It takes ~3 minutes on my computer)
"""
results = objExp.simulation(total_time = 24, timeUnit = "hr", courant = 0.8)

### 4.3 Total CO2 mass
The figure below shows the change in the total amount of CO2 emitted by airplanes over the simulation time. The total amount of CO2 increases linearly with time. At the final simulation moment, which is at 88,382 seconds, the total mass of CO2 is 193,558.58 kilograms. This results in an emission rate of exactly 2.19 kg/s, which corresponds to the emission rate set in section 4.1.1.

In [ ]:
mass_timeList = results["var_ext_dict"]["co2"].sum(axis = 1)
timeList = results["timeList"]
fig = plt.figure()
ax = fig.add_subplot(1, 1, 1)
ax.plot(timeList / 3600, mass_timeList, color = "red")
ax.grid(linestyle = "dashed")
ax.set_xlabel("Time (hour)")
ax.set_ylabel("Mass (Kg)")
ax.set_title("Total CO2 mass")
plt.show()
print("Last time = %d (s), total CO2 mass = %f (Kg), average emission rate = %f (Kg/s)" % (timeList[-1], mass_timeList[-1], mass_timeList[-1]/timeList[-1]))

### 4.4 Results visualization

In [ ]:
"""
    The function searches for the nearest simulation results corresponding to a specific time.
"""
def search_nearest_itime(results, time):
    return np.argmin(np.abs(results["timeList"] - time))

In [ ]:
"""
    Draw the airplane emitted CO2 distribution at a specific time and save as a png file.
"""
def draw_one_time_for_ae(time, results, outName):
    
    cmap = get_colormap("YlOrRd")

    final_co2 = results["var_show_dict"]["co2"][search_nearest_itime(results, time)]
    co2_max = np.array([co2.max() for co2 in results["var_show_dict"]["co2"]]).max()

    fig = objExp.objMesh.plot_data(
        data = final_co2 * 1e9,
        #colorscale = "jet",
        #colorscale = "Greens",
        colorscale = cmap,
        opacity = 1,
        showscale = True,
        cmin = 0,
        cmax = co2_max * 1e9 / 2,
        #cmax = 50,
    )
    objExp.objMesh.implement_coastlines(dataDir + "/coastlines.json")
    fig = objExp.objMesh.plot_coastlines(fig, threshold = 10, color = "black")
    fig = objExp.objMesh.plot_gridlines(fig)
    #fig.write_image(plotDir + "/airplane_emission.png", width=960, height=540, scale=2)
    path1_lon, path1_lat = objExp.objScn.get_great_circle_path()
    path2_lon, path2_lat = objExp.objScn.get_great_circle_path_until_time(time)
    fig = objExp.objMesh.plot_line(fig, path1_lon, path1_lat, color = "blue", dash = "dash", width = 1)
    fig = objExp.objMesh.plot_line(fig, path2_lon, path2_lat, color = "blue", dash = "solid", width = 1)
    fig.update_layout(title = "CO2 concentration (µg/m^3); time = " + str(time / 3600) + " hr", margin=dict(l=0, r=0, t=25, b=0))  # 设置左、右、顶、底的边距
    fig.write_image(outName, width=960, height=540, scale=3)
    print("Image is saved to " + outName)

In [ ]:
"""
    Plot the airplane emitted CO2 simulation result at each hour.
    (It may take a little while, ~8 minutes on my computer)
"""
hourPlots = np.arange(25) #[6, 12, 18, 24]
for hour in hourPlots:
    print("Drawing hour = ", hour, " ...")
    time = hour * 3600
    outName = plotDir + "/airplane_emission_t_" + str(hour).zfill(3) + ".png"
    draw_one_time_for_ae(time, results, outName)
print("!! Complete !!")

#### 4.4.1 The animation of airplane emitted CO2 simulation result

The concentration of CO2 emitted by the airplane is indicated by a red-to-yellow scale, the solid blue line represents the route the airplane has already flown, and the dashed blue line represents the route the airplane will fly.

In [ ]:
"""
    Plot the airplane emitted CO2 simulation result at each hour.
"""
hourPlots = np.arange(25)
images = [PIL_Image.open(plotDir + "/airplane_emission_t_" + str(hour).zfill(3) + ".png") for hour in hourPlots]
images[0].save(plotDir + '/airplane_emission.gif', save_all=True, append_images=images[1:], optimize=False, duration=500, loop=0)
Image(url=plotDir + '/airplane_emission.gif')

#### 4.4.2 The 3D plot for the final CO2 distribution

This 3D display allows you to see the distribution of CO2 globally from various angles (not just one angle). By rotating the 3D view below, you can see that the CO2 emitted by the airplane has drifted to the western Pacific along the jet stream.


In [ ]:
hour = 24

final_co2 = results["var_show_dict"]["co2"][search_nearest_itime(results, hour * 3600)]

cmap = get_colormap("YlOrRd")

fig = objExp.objMesh.plot_data(
    data = final_co2 * 1e9,
    colorscale = cmap,
    opacity = 1,
    showscale = True,
    cmin = 0,
    cmax = final_co2.max() *1e9 / 2,
)

objExp.objMesh.implement_coastlines(dataDir + "/coastlines.json")
fig = objExp.objMesh.plot_coastlines(fig, threshold = 10, color = "black")
fig = objExp.objMesh.plot_gridlines(fig)
fig.update_layout(title = "CO2 concentration (µg/m^3)")
path1_lon, path1_lat = objExp.objScn.get_great_circle_path()
path2_lon, path2_lat = objExp.objScn.get_great_circle_path_until_time(time)
fig = objExp.objMesh.plot_line(fig, path1_lon, path1_lat, color = "blue", dash = "dash", width = 1)
fig = objExp.objMesh.plot_line(fig, path2_lon, path2_lat, color = "blue", dash = "solid", width = 1)
fig.show()

### 4.5 Conclusion

This section implements the simulation of CO2 concentration advection due to airplane emissions. The airplane flies westward, and the CO2 plume advects eastward with the westerlies, covering almost the entire Eastern Hemisphere. The simulation results show that the rate of increase in CO2 mass over time matches the emission rate of the airplane, indicating that the advection scheme satisfies mass conservation.

The results presented in this section confirm that the last objective outlined in the introduction has been achieved: __"Model the distribution of CO2 concentrations emitted during an airplane's flight."__

## 5. Reflection

### 5.1 My Learnings

Through this project, I have gained a lot of geometric knowledge about polyhedra and learned how to construct spherical triangle and hexagon meshes at various resolutions. During the development and debugging of the advection algorithms, I gained a deeper understanding of the finite volume method, including its differences from the finite difference method. For example, I understood why the upwind scheme is used in flux calculations, utilizing the values from the upwind region as the boundary values between two tails, instead of using the average values of the two tails. I learned how the upwind scheme prevents abnormally high values in areas of shock and how the finite volume method ensures mass conservation. I also understood how dissipation phenomena occur and why higher resolution and higher Courant numbers (within stable conditions) can reduce dissipation. Additionally, I learned how to use matrix operations to improve program efficiency, and my skills in object-oriented programming have improved.

### 5.2 Limitations

1. The algorithm used in this project is a first-order finite volume scheme, which shows significant dissipation, resulting in false diffusion.

2. The algorithm cannot guarantee non-negative results, and negative results may occur when simulating low concentration objects in strong diverging wind fields (for instance, in the "Divergence_Block" ideal experiment, which is not displayed in the mini-app; see details in src/ideal_scenarios.py, class Scenario_Divergence_Block).

3. The temperature advection simulation in Section 3 can model the main dynamic factors of temperature field evolution, but for actual weather forecasting or climate prediction, it is necessary to compute in 3D space across multiple atmospheric layers and consider vertical atmospheric movements and additional heat sources and sinks.

### 5.3 Future Work

1. Use higher-order finite volume schemes to solve advection equations and mitigate dissipation.

2. Integrate multi-layer grids and model more complex phenomena such as advection, diffusion, and turbulence in three-dimensional space based on the Navier-Stokes equations.